In [0]:
df = spark.read.format("csv").option("header", "true").load("/Volumes/external_catalog/default/external_volume/Employee_Attrition.csv")
display(df)

In [0]:
from pyspark.sql.functions import col

#Filter for high risk attrition employees
high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction").cast("int") < 3)
)

#Select relevant informative columns
selected_columns = [
    "EmployeeNumber",
    "Department",
    "JobRole",
    "JobSatisfaction",
    "Age",
    "Gender",
    "MaritalStatus",
    "MonthlyIncome",
    "OverTime",
    "YearsAtCompany"
]

high_risk_df = high_risk_df.select(selected_columns)
display(high_risk_df)

#Write to Delta table 
high_risk_df.write.format("delta").mode("overwrite").saveAsTable("external_catalog.default.high_risk_attrition_employees")

In [0]:
history_df =spark.sql("DESCRIBE HISTORY external_catalog.default.high_risk_attrition_employees")
display(history_df.select("version","timestamp","operation")) 

In [0]:
delta_df = spark.read.format("delta").table("external_catalog.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
from pyspark.sql import Row

dummy_record = Row(
    EmployeeNumber="99999",
    Department="DummyDept",
    JobRole="DummyRole",
    JobSatisfaction="1",
    Age="30",
    Gender="Male",
    MaritalStatus="Single",
    MonthlyIncome="1000",
    OverTime="No",
    YearsAtCompany="0"
)

dummy_df = spark.createDataFrame([dummy_record])

dummy_df.write.format("delta").mode("append").saveAsTable("external_catalog.default.high_risk_attrition_employees")

In [0]:
delta_df = spark.read.format("delta").table("external_catalog.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
history_df =spark.sql("DESCRIBE HISTORY external_catalog.default.high_risk_attrition_employees")
display(history_df.select("version","timestamp","operation")) 

In [0]:
delta_df_v = spark.read.format("delta").option("versionAsOf", 0).table("external_catalog.default.high_risk_attrition_employees")
display(delta_df_v)